In [1]:
import pandas as pd

In [2]:
data = pd.read_csv(r"C:\Users\Arya\Desktop\data_science\ICT\Assignment5\mobile_price_classification.csv")

# read the data and save it into a variable "data".

In [3]:
data.head()

# printing the head of the data.

,battery_power,bluetooth,clock_speed,dual_sim,front_cam,4G,int_memory,m_dep,mobile_wt,n_cores,...,px_height,px_width,ram,sc_h,sc_w,talk_time,three_g,touch_screen,wifi,price_range
0,842,0,2.2,0,1,0,7,0.6,188,2,...,20,756,2549,9,7,19,0,0,1,1
1,1021,1,0.5,1,0,1,53,0.7,136,3,...,905,1988,2631,17,3,7,1,1,0,2
2,563,1,0.5,1,2,1,41,0.9,145,5,...,1263,1716,2603,11,2,9,1,1,0,2
3,615,1,2.5,0,0,0,10,0.8,131,6,...,1216,1786,2769,16,8,11,1,0,0,2
4,1821,1,1.2,0,13,1,44,0.6,141,2,...,1208,1212,1411,8,2,15,1,1,0,1


In [4]:
list(data.columns)

# list of column names.

['battery_power',
 'bluetooth',
 'clock_speed',
 'dual_sim',
 'front_cam',
 '4G',
 'int_memory',
 'm_dep',
 'mobile_wt',
 'n_cores',
 'primary_camera',
 'px_height',
 'px_width',
 'ram',
 'sc_h',
 'sc_w',
 'talk_time',
 'three_g',
 'touch_screen',
 'wifi',
 'price_range']

In [5]:
import numpy as np                                   # A numerical computing library
from sklearn.model_selection import train_test_split # To evaluate model performance on unseen data.
from sklearn.preprocessing import StandardScaler     # Standardizes features.
from sklearn.metrics import classification_report    # To evaluate model performance.
import tensorflow as tf                              # Deep Learning framework
from tensorflow import keras                         # High-level API inside TensorFlow.
from tensorflow.keras import layers                  # Used to add layers to neural networks.

# import all the libraries.

In [6]:
X = data.drop("price_range", axis=1)
y = data["price_range"]

# Split features and target.

In [7]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [8]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Scaling the features.

In [9]:
import warnings
warnings.filterwarnings("ignore")

In [10]:
model = keras.Sequential([
    layers.Dense(128, activation='relu', input_shape=(X_train.shape[1],)),
    layers.Dense(64, activation='relu'),
    layers.Dense(32, activation='relu'),
    layers.Dense(4, activation='softmax')  # 4 classes
])

# Build ANN model.

In [11]:
model.compile(optimizer='adam',loss='sparse_categorical_crossentropy',metrics=['accuracy'])

# Compile model.

In [12]:
history = model.fit(X_train, y_train,epochs=5,batch_size=32,validation_split=0.2)

# Train model

Epoch 1/5
40/40 ━━━━━━━━━━━━━━━━━━━━ 5s 10ms/step - accuracy: 0.4328 - loss: 1.2419 - val_accuracy: 0.5875 - val_loss: 1.0575
Epoch 2/5
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7234 - loss: 0.7601 - val_accuracy: 0.7812 - val_loss: 0.6124
Epoch 3/5
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8664 - loss: 0.4209 - val_accuracy: 0.8938 - val_loss: 0.3572
Epoch 4/5
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9250 - loss: 0.2666 - val_accuracy: 0.9219 - val_loss: 0.2639
Epoch 5/5
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9508 - loss: 0.1866 - val_accuracy: 0.9219 - val_loss: 0.2290


In [13]:
loss, accuracy = model.evaluate(X_test, y_test)
print("Test Accuracy:", accuracy)

# Evaluate testing data.

13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9125 - loss: 0.2331  
Test Accuracy: 0.9125000238418579


In [14]:
y_pred = np.argmax(model.predict(X_test), axis=1)
print(classification_report(y_test, y_pred))

# Predictions

13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step 
              precision    recall  f1-score   support

           0       0.95      0.93      0.94       105
           1       0.87      0.88      0.87        91
           2       0.86      0.90      0.88        92
           3       0.96      0.93      0.95       112

    accuracy                           0.91       400
   macro avg       0.91      0.91      0.91       400
weighted avg       0.91      0.91      0.91       400



# Hyperparameter Tuning

In [15]:
import keras_tuner as kt

In [16]:
def build_model(hp):
    model = keras.Sequential()

    # Tune number of neurons
    model.add(layers.Dense(
        units=hp.Int('units1', min_value=32, max_value=256, step=32),
        activation='relu',
        input_shape=(X_train.shape[1],)
    ))

    model.add(layers.Dense(
        units=hp.Int('units2', min_value=32, max_value=128, step=32),
        activation='relu'
    ))

    model.add(layers.Dense(4, activation='softmax'))

    model.compile(
        optimizer=keras.optimizers.Adam(
            hp.Choice('learning_rate', [0.01, 0.001, 0.0001])
        ),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    return model


In [17]:
tuner = kt.RandomSearch(
    build_model,
    objective='val_accuracy',
    max_trials=5,
    executions_per_trial=1,
    directory='tuner_dir',
    project_name='mobile_price')

Reloading Tuner from tuner_dir\mobile_price\tuner0.json


In [18]:
tuner.search(X_train, y_train, epochs=30, validation_split=0.2)

In [19]:
best_model = tuner.get_best_models(num_models=1)[0]

In [20]:
loss, accuracy = best_model.evaluate(X_test, y_test)
print("Tuned Test Accuracy:", accuracy)

13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9300 - loss: 0.1633  
Tuned Test Accuracy: 0.9300000071525574
